<a href="https://colab.research.google.com/github/JasmineLee7/movie-rec/blob/main/notebooks/04_what_it_learned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Milestone 4: What Did It Learn, and What Would It Recommend to You?

#Part 1: Set up

In [1]:
import os
import subprocess
import sys

USER = 'JasmineLee7'
REPO = 'movie-rec'

if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone', f'https://github.com/{USER}/{REPO}.git', REPO],
        check=True)

sys.path.insert(0, f'{REPO}/src')
from evaluate import to_matrix
from models import baseline_matrix, rank_k

In [2]:
import urllib.request
import zipfile
import numpy as np
import pandas as pd

if not os.path.exists('ml-100k'):
    urllib.request.urlretrieve(
        'https://files.grouplens.org/datasets/movielens/ml-100k.zip',
        'ml-100k.zip')
    zipfile.ZipFile('ml-100k.zip').extractall()

ratings = pd.read_csv(
    'ml-100k/u.data', sep='\t',
    names=['user', 'item', 'rating', 'time'])

In [3]:
titles = pd.read_csv("ml-100k/u.item", sep="|", encoding="latin-1",
                     header=None, usecols=[1])[1].to_numpy()
print(len(titles), titles[49])

1682 Star Wars (1977)


#Part 2 Fit Model 2 on all the ratings

In [4]:
R = to_matrix(ratings)
known = ~np.isnan(R)
B = baseline_matrix(R)
X = np.where(known, R - B, 0.0)
U, s, Vt = np.linalg.svd(X, full_matrices=False)
n_ratings = known.sum(axis=0)
popular = n_ratings >= 50
print("popular movies:", popular.sum())

popular movies: 603


This builds the full rating matrix using all available ratings (rather than holding out a test set like in Milestone 3), computes the SVD on the baseline-subtracted residuals, and creates a boolean mask for the 603 "popular" movies that have at least 50 ratings so obscure movies with only a few ratings don't skew the results.

# Part 3: What did it learn? Naming the hidden factors

In [5]:
def show_factor(f, n=6):
    """Popular movies at the two ends of hidden factor f (f = 1, 2, ...)."""
    v = Vt[f - 1]
    idx = np.where(popular)[0]
    order = idx[np.argsort(v[idx])]
    print(f"Factor {f}, one end:")
    for m in order[:n]:
        print(f" {v[m]:+.3f} {titles[m]}")
    print(f"Factor {f}, other end:")
    for m in order[::-1][:n]:
        print(f" {v[m]:+.3f} {titles[m]}")

show_factor(1)

Factor 1, one end:
 -0.087 Clockwork Orange, A (1971)
 -0.079 Trainspotting (1996)
 -0.072 Leaving Las Vegas (1995)
 -0.072 Pulp Fiction (1994)
 -0.069 Brazil (1985)
 -0.067 Fargo (1996)
Factor 1, other end:
 +0.211 Independence Day (ID4) (1996)
 +0.175 Return of the Jedi (1983)
 +0.149 Raiders of the Lost Ark (1981)
 +0.149 Back to the Future (1985)
 +0.136 Braveheart (1995)
 +0.134 Forrest Gump (1994)


This function looks at one hidden factor, represented by a row of `Vt`. Since Python starts counting at 0, factor `f` is row `f - 1`. It keeps only movies with at least 50 ratings, sorts their factor scores, and prints the six lowest and six highest scores with their titles. These scores are factor coordinates, not predicted star ratings. Comparing the two groups helps us describe the preference pattern that the SVD found.

**Factor 1: Dark or unconventional films vs. mainstream crowd-pleasers.** The interpretation below follows the assignment's reference output; run Parts 1-3 to confirm the movie groups in this notebook. Films such as *A Clockwork Orange*, *Trainspotting*, and *Pulp Fiction* appear together at one end, while *Independence Day*, *Return of the Jedi*, and *Back to the Future* appear at the other. A reasonable interpretation is that this factor separates darker, more unconventional films from mainstream adventures and crowd favorites. This is our description of the pattern, not a genre label given to the model: the SVD used rating patterns only. The signs can flip between SVD runs, so the groups matter more than which side is positive.

In [6]:
show_factor(3)

Factor 3, one end:
 -0.137 Independence Day (ID4) (1996)
 -0.131 Star Trek: First Contact (1996)
 -0.130 Rock, The (1996)
 -0.129 Alien (1979)
 -0.121 Monty Python and the Holy Grail (1974)
 -0.119 Aliens (1986)
Factor 3, other end:
 +0.163 Sound of Music, The (1965)
 +0.144 Grease (1978)
 +0.099 E.T. the Extra-Terrestrial (1982)
 +0.087 Mary Poppins (1964)
 +0.085 Jaws 2 (1978)
 +0.084 Snow White and the Seven Dwarfs (1937)


This calls the same function for factor 3, using the SVD already calculated in Part 2. It prints popular movies at the two ends of the third factor so we can compare a different preference pattern without fitting the model again.

**Factor 3: Science fiction and action vs. musicals and family films.** The interpretation below follows the assignment's reference output; run Parts 1-3 to confirm the movie groups in this notebook. Movies such as *Alien*, *Aliens*, and *Star Trek: First Contact* appear at one end, while *The Sound of Music*, *Grease*, and *Mary Poppins* appear at the other. This suggests a contrast between science fiction/action and musicals/family films. The split is not perfect: a factor can combine several patterns, and some movies do not fit a simple name. These interpretations describe associations in the ratings, not proof that every user likes or dislikes an entire genre. As with factor 1, the signs may be reversed without changing the pattern.

# Part 4: What would it recommend to you?

In [7]:
def find(word):
    """Print the ID and title of every movie whose title contains word."""
    for m, t in enumerate(titles):
        if word.lower() in t.lower():
            print(m + 1, t)

find("alien")

176 Aliens (1986)
183 Alien (1979)
343 Alien: Resurrection (1997)
665 Alien 3 (1992)


This function searches the dataset for movies based on words in their titles. It works regardless of capitalization and shows each matching movie with its ID. These IDs are then used to add our own movie ratings.

In [8]:
my_ratings = {
    50: 5,
    172: 5,
    183: 5,
    195: 5,
    89: 4,
    1: 4,
    143: 1,
    451: 1,
    88: 1,
    739: 2
}

We assigned ratings from 1 to 5 to different movies and stored them in a dictionary using their movie IDs. This gives the model information about our preferences that it can use to recommend other movies.

In [9]:
me = np.full((1, R.shape[1]), np.nan)

for movie, stars in my_ratings.items():
    me[0, movie - 1] = stars

R_me = np.vstack([R, me])
print(R_me.shape)

known_me = ~np.isnan(R_me)
B_me = baseline_matrix(R_me)
X_me = np.where(known_me, R_me - B_me, 0.0)

U, s, Vt = np.linalg.svd(X_me, full_matrices=False)
P = rank_k(U, s, Vt, 10) + B_me

(944, 1682)


To include the new user in the recommendation system, we combined their ratings with the original data. We then applied SVD to identify rating patterns and generate predictions. Following our results from Milestone 3, we kept the rank at 10.

In [10]:
mine = P[-1]
base = B_me[-1]
can_pick = popular & np.isnan(R_me[-1])

def top(scores, n=10):
    """The n highest-scoring movies recommended."""
    return np.argsort(-np.where(can_pick, scores, -np.inf))[:n]

print("Model 2 recommends:")
for m in top(mine):
    print(f" {mine[m]:.2f} {titles[m]}")

print("Baseline recommends:")
for m in top(base):
    print(f" {base[m]:.2f} {titles[m]}")

print("in both lists:", len(set(top(mine)) & set(top(base))))

Model 2 recommends:
 4.28 Close Shave, A (1995)
 4.26 Wrong Trousers, The (1993)
 4.20 Fargo (1996)
 4.15 Casablanca (1942)
 4.14 Raiders of the Lost Ark (1981)
 4.11 Wallace & Gromit: The Best of Aardman Animation (1996)
 4.08 Godfather, The (1972)
 4.07 Schindler's List (1993)
 4.07 Shawshank Redemption, The (1994)
 4.06 Return of the Jedi (1983)
Baseline recommends:
 4.23 Close Shave, A (1995)
 4.15 Wrong Trousers, The (1993)
 4.13 Schindler's List (1993)
 4.13 Shawshank Redemption, The (1994)
 4.09 Casablanca (1942)
 4.08 Wallace & Gromit: The Best of Aardman Animation (1996)
 4.08 Usual Suspects, The (1995)
 4.06 Good Will Hunting (1997)
 4.01 Titanic (1997)
 4.01 12 Angry Men (1957)
in both lists: 6


We used both Model 2 and the baseline to identify the top 10 movie recommendations for the user, excluding movies they already rated. By comparing the two lists, we can determine how many recommendations they share and whether the models suggest similar movies.

In [11]:
personal = mine - base

print("Moved up most for me:")
for m in top(personal, 6):
    print(f" {personal[m]:+.2f} {titles[m]}")

print("Moved down most for me:")
for m in top(-personal, 6):
    print(f" {personal[m]:+.2f} {titles[m]}")

Moved up most for me:
 +0.34 Fargo (1996)
 +0.34 Return of the Jedi (1983)
 +0.25 Star Trek: The Wrath of Khan (1982)
 +0.22 Star Trek: First Contact (1996)
 +0.21 2001: A Space Odyssey (1968)
 +0.20 Raiders of the Lost Ark (1981)
Moved down most for me:
 -0.36 English Patient, The (1996)
 -0.21 Liar Liar (1997)
 -0.19 Dirty Dancing (1987)
 -0.19 Mr. Holland's Opus (1995)
 -0.18 Ghost (1990)
 -0.17 Mrs. Doubtfire (1993)


We compared Model 2's predictions with the baseline to see how personalization changed the movie ratings. The differences help identify which movies the model ranked higher or lower based on the user's preferences.